### ULTIMO PERIODO ESTADISTICA UNICAMENTE POR PERIODO ESTUDIANTE

In [1]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
import re

#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)
    

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

     SELECT DISTINCT *
FROM CUN.ESTADISTICA_ESTUDIANTE_2
WHERE COD_PERIODO NOT LIKE '%I%' AND COD_PERIODO NOT LIKE '%M%' AND NOM_UNIDAD != 'EDUCACION CONTINUADA'

--AND COD_PERIODO LIKE '%A%' OR COD_PERIODO LIKE '%B%' OR COD_PERIODO LIKE '%V%' 

--WHERE cod_periodo LIKE '%A%'
  -- OR cod_periodo LIKE '%B%'
   --OR cod_periodo LIKE '%V%'
   --OR cod_periodo LIKE '%E%'

        
        
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_est = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

Conexion a la base de datos realizada


In [2]:
df_est['COD_PERIODO'] = df_est['COD_PERIODO'].astype(str)

In [3]:
df_est['COD_PERIODO'].unique()

array(['017E1', '017E2', '017V1', '017V2', '017V3', '017V4', '017V5',
       '017V6', '018E1', '018E3', '018E4', '019E1', '019E2', '019E3',
       '019E4', '020E1', '18V01', '18V03', '18V04', '18V05', '18V06',
       '18V02', '19V01', '19EQ1', '19V02', '19V03', '19V04', '19V05',
       '19V06', '20123', '2012A', '2012B', '2013A', '2013B', '20142',
       '20143', '20144', '2014A', '2014B', '2014C', '20151', '20152',
       '2015A', '2015B', '2014O', '2015E', '20161', '20162', '2016A',
       '2016B', '2016E', '2016V', '20171', '2017A', '2017B', '2017E',
       '2018A', '2018B', '2019A', '2019B', '2019C', '2019J', '2019Q',
       '2020A', '2020B', '2018E', '2020D', '2021A', '2021B', '2021T',
       '2022A', '2022B', '2022E', '2022J', '2022T', '2023A', '2023B',
       '2023J', '2023T', '2024A', '2024B', '2024J', '2024T', '2025A',
       '2025B', '2025C', '2025Q', '2025T', '20D02', '20EQ3', '20ET2',
       '20ET3', '20ET4', '20T05', '20T06', '20V01', '20V02', '20V03',
       '20V04', '202

In [4]:
df_est2 = df_est.sort_values(by=['COD_PERIODO'])

### AGRUPACIÓN PARA TRAER UN REGISTRO

In [5]:
import re
def extraer_anio_periodo(cod):
    match = re.match(r"(\d+)([A-Z]+)(\d+)?", cod)  # Extrae números, letras y números opcionales
    if match:
        anio = match.group(1)  # Extrae el año como string
        letra = match.group(2)  # Extrae las letras del periodo
        num_periodo = int(match.group(3)) if match.group(3) else 0  # Extrae el número del periodo
        
        return anio, letra, num_periodo
    return None, None, None

# Aplicar la función para extraer año y período
df_est2[['AÑO', 'LETRA', 'NUM_PERIODO']] = df_est2['COD_PERIODO'].apply(lambda x: pd.Series(extraer_anio_periodo(x)))

In [6]:
# Ordena por identificacion, AÑO, NUM_PERIODO en orden adecuado
# Se da prioridad primero al año, luego al número del periodo y luego a la letra
df_sorted = df_est2.sort_values(by=['NUM_IDENTIFICACION', 'AÑO', 'NUM_PERIODO', 'LETRA'], ascending=[True, False, False, True])

In [7]:
# Obtiene el último período para cada identificacion
df_ultimo_periodo = df_sorted.groupby('NUM_IDENTIFICACION').first().reset_index()


In [8]:
df_total_sorted2 = df_sorted[['NUM_IDENTIFICACION','COD_PERIODO','NOM_UNIDAD']]
df_total_sorted2[df_total_sorted2['NUM_IDENTIFICACION']=='1018477214']

,NUM_IDENTIFICACION,COD_PERIODO,NOM_UNIDAD
847613,1018477214,25P02,INGENIERIA DE SISTEMAS
773940,1018477214,24P06,INGENIERIA DE SISTEMAS
772049,1018477214,24P04,INGENIERIA DE SISTEMAS
951281,1018477214,24V02,INGENIERIA DE SISTEMAS


In [9]:
df_ultimo_periodo[df_ultimo_periodo['NUM_IDENTIFICACION']=='1018477214']

,NUM_IDENTIFICACION,COD_PERIODO,NOM_TERCERO,SEG_NOMBRE,PRI_APELLIDO,SEG_APELLIDO,GEN_TERCERO,TIP_IDENTIFICACION,DIR_EMAIL_PER,DIR_EMAIL,...,SECTOR_EMPRESA,EMPRESA,DESCRIPCION_RELACION_TECNOLOGIA,METODO_FINANCIAMIENTO,CANAL_INFORMACION_CONOCIO_CUN,ULTIMO_NIVEL_EDUCAT_COMPLETADO,PRINCIPAL_MOTIVACION_CONTINUAR_ESTUDIOS,INST_EDUCATIVA_CONSIDERO_ANTES_QUE_CUN,LETRA,NUM_PERIODO
67355,1018477214,25P02,EDWARD,STEVEN,MORENO,AVENDAÑO,M,C,stevenmoreno.sm9@gmail.com,edward.morenoa@cun.edu.co,...,None,None,Excelente,Financiamiento propio,Recomendación de un amigo o familiar,Tecnólogo,Mejorar las oportunidades laborales,Si,P,2.0


In [10]:
df_ultimo_periodo.columns

Index(['NUM_IDENTIFICACION', 'COD_PERIODO', 'NOM_TERCERO', 'SEG_NOMBRE',
       'PRI_APELLIDO', 'SEG_APELLIDO', 'GEN_TERCERO', 'TIP_IDENTIFICACION',
       'DIR_EMAIL_PER', 'DIR_EMAIL', 'TEL_RESIDENCIA', 'TEL_CELULAR',
       'COD_UNIDAD', 'NOM_UNIDAD', 'COD_PENSUM', 'COD_PENSUM_LAURA',
       'NOM_DEPENDENCIA', 'MUNICIPIO_RESIDENCIA', 'DEPARTAMENTO_RESIDENCIA',
       'MUNICIPIO_SEDE', 'DEPARTAMENTO_SEDE', 'FEC_NACIMIENTO', 'EDAD',
       'MODALIDAD', 'NIVEL_FORMACION', 'NOM_SECCIONAL', 'COD_DPTO', 'CICLO',
       'ubicacion', 'CLASE_ACTUAL', 'NUEVO', 'ESTADO_ALUMNO', 'ESTADO_PAGO',
       'UNIDADNEGOCIO', 'SEMESTRE', 'NOM_SEDE', 'DEPARTAMENTO_REGIONAL',
       'ESTADO_MOODLE', 'ultimo_acceso_moodle', 'AÑO', 'SEMESTRE_CALCULADO',
       'NOMBRE_ESCUELA', 'Estrato', 'promedio', 'PRO_ACUMULADO',
       'ESTADO_CIVIL', 'ESTADO_CIVIL_ACTUAL', 'RANGO_EDAD',
       'FECHA_EXP_CEDULA_ACTUALIZADO', 'TELEFONO_ACTUALIZADO',
       'TIENE_DISCAPACIDAD', 'DISCAPACIDAD', 'OTRA_DISCAPACIDAD',
     

In [11]:
df_ultimo_periodo.shape

(251520, 76)

In [12]:
df_ultimo_periodo.drop(columns={'LETRA','NUM_PERIODO'}, inplace=True)

In [13]:
##ultimo_periodo.to_csv('ultimo_periodo_est.csv')

In [19]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'ULTIMO_PERIODO_ESTADISTICA'
schema_name = 'CUN'
# Send the data to sql server
df_ultimo_periodo.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()